# Indian B2B Road-Freight — Real-Data Rebuild & Model Export (Local)

End-to-end: real weather (Open-Meteo) + real diesel (PPAC/IOC) → build 60k dataset →
train Model 1 (freight should-cost) & Model 2 (invoice-risk) → **export model files** →
SARIMA forecaster. Run the cells top to bottom.

## 1. Project Directory Setup

In [1]:
import os
PROJECT = os.path.abspath(os.getcwd())
if not os.path.exists(os.path.join(PROJECT, 'code')) and os.path.exists(os.path.join(PROJECT, '..', 'code')):
    PROJECT = os.path.abspath(os.path.join(PROJECT, '..'))
print('Project Root:', PROJECT)

Project Root: d:\chirag\bundle


## 2. Verify Project Structure

In [2]:
import os
print('Available files in root:', sorted(os.listdir(PROJECT)))
assert os.path.exists(os.path.join(PROJECT, 'code')), f'Not found: code directory in {PROJECT}'

Available files in root: ['.env', '.venv', 'Colab_RealBuild.ipynb', 'DATA_QUALITY_NOTES.md', 'README.md', 'README_EXECUTION.md', 'REPORT.md', 'chi_freight_models.zip', 'code', 'data', 'extraction_scripts', 'real_cache', 'requirements.txt']


## 3. Install dependencies

In [3]:
import os
req_file = os.path.join(PROJECT, 'requirements.txt')
%pip install -q -r "$req_file"
import xgboost, sklearn, statsmodels, pandas
print('xgboost', xgboost.__version__, '| sklearn', sklearn.__version__,
      '| statsmodels', statsmodels.__version__, '| pandas', pandas.__version__)

Note: you may need to restart the kernel to use updated packages.
xgboost 3.4.1 | sklearn 1.9.0 | statsmodels 0.15.0 | pandas 3.0.5


## 4. (Optional) Re-fetch the REAL data yourself
The zip already ships real caches in `real_cache/` (weather + diesel), so you can
**skip 4a/4b**. Route distance is different: it still defaults to the modeled
haversine × circuity stand-in unless you run **4c** below, since real road
distances were never bundled pre-built.

**4a. Real weather** — Open-Meteo Archive, no key, ~2 min (18 hubs, resumable).

In [4]:
import os
os.chdir(os.path.join(PROJECT, 'extraction_scripts'))
!python run_openmeteo.py --hubs ../data/hubs.csv --out ../real_cache/weather_cache.csv \
    --start 2023-01-01 --end 2026-01-31 --sleep 0.8

Resuming: 18 hubs already cached -> ['HUB01', 'HUB02', 'HUB03', 'HUB04', 'HUB05', 'HUB06', 'HUB07', 'HUB08', 'HUB09', 'HUB10', 'HUB11', 'HUB12', 'HUB13', 'HUB14', 'HUB15', 'HUB16', 'HUB17', 'HUB18']
DONE. 2916 week-rows across 18 hubs -> ../real_cache/weather_cache.csv


**4b. Real diesel** — PPAC/IOC daily RSP (no key), normalised to 13 states.

In [5]:
import os, requests
os.chdir(os.path.join(PROJECT, 'extraction_scripts'))
url='https://raw.githubusercontent.com/realblackcross/DashRSP/main/public/rspData.csv'
os.makedirs('../real_cache', exist_ok=True)
open('../real_cache/ppac_rsp_raw.csv','wb').write(requests.get(url,timeout=60).content)
!python load_ppac_diesel.py --infile ../real_cache/ppac_rsp_raw.csv \
    --out ../real_cache/fuel_prices.csv --start 2023-01-01 --end 2025-12-31

Wrote 14248 (date,state) diesel rows to ../real_cache/fuel_prices.csv
  window 2023-01-01..2025-12-31, 13 states
  DIRECT-REAL states (4): ['Delhi', 'Maharashtra', 'Tamil Nadu', 'West Bengal']
  DERIVED states: ['Andhra Pradesh', 'Gujarat', 'Haryana', 'Karnataka', 'Madhya Pradesh', 'Punjab', 'Rajasthan', 'Telangana', 'Uttar Pradesh']
  national real backbone range: 90.17..92.72 INR/L


**4c. Real route distance** — OpenRouteService (ORS), 153 combinations, ~3-4 min.
Needs a free API key (no card): sign up at https://openrouteservice.org/dev/#/signup
and paste it when prompted below. Writes to `real_cache/`, matching 4a/4b's staging
convention (this is what Step 5 will read from).

In [6]:
import os
from getpass import getpass
if not os.environ.get('ORS_API_KEY'):
    os.environ['ORS_API_KEY'] = getpass('Enter your OpenRouteService API key: ')
os.chdir(os.path.join(PROJECT, 'extraction_scripts'))
!python run_ors.py --hubs ../data/hubs.csv --out ../real_cache/route_distance_cache.csv

Fetching 153 unique hub-pair routes from ORS (profile=driving-hgv)...
  20/153 routes fetched (20 ok, 0 dropped)
  40/153 routes fetched (40 ok, 0 dropped)
  60/153 routes fetched (60 ok, 0 dropped)
  80/153 routes fetched (80 ok, 0 dropped)
  100/153 routes fetched (100 ok, 0 dropped)
  120/153 routes fetched (120 ok, 0 dropped)
  140/153 routes fetched (140 ok, 0 dropped)
  153/153 routes fetched (153 ok, 0 dropped)

Wrote 153 routes to ../real_cache/route_distance_cache.csv; dropped 0 pairs.


## 5. Build the 60k dataset on REAL data
Consumes the real weather + fuel + route-distance caches (0 fallback = full real
coverage). If you skipped 4c, omit `--route-cache` and the pipeline falls back to
modeled haversine × circuity distances automatically — the build still runs either
way. Writes the model-ready tables into `code/output/`.

In [7]:
import os
os.chdir(os.path.join(PROJECT, 'code'))
!python build_dataset.py --orders 60000 --seed 42 --outdir output \
    --weather-cache ../real_cache/weather_cache.csv \
    --fuel-cache ../real_cache/fuel_prices.csv \
    --route-cache ../real_cache/route_distance_cache.csv

[weather] REAL cache -> ../real_cache/weather_cache.csv (20,286 hub-days, 18 hubs)
[build] Phase 1: hubs / vendors / cost assumptions
[build] Phase 3: route_distance_cache (unique hub pairs)
[routing] REAL cache -> ../real_cache/route_distance_cache.csv (153 routes)
[build]         153 unique routes cached
[build] Phase 1: order skeleton
[build] Phase 2: cargo weight/volume + dimensional weight
[build] Phase 3 join: distance / ideal_days onto orders
[build] Phase 6: quoted_days
[build] Phase 4: expected weather severity (planned window)
[build] Phase 5: fuel_prices + merge_asof(by=state, backward, 7D)
[fuel] REAL cache -> ../real_cache/fuel_prices.csv (14,248 rows, 13 states, 2023-01-01..2025-12-31)
[build]         fuel-price fallback applied to 0 rows
[build] Phase 7: base_freight_cost label (+/-2% noise)
[build] Phase 9a: anomaly injection (5-8%, three mechanisms)
[build]         anomaly_rate=0.056  positives=3370
[build] Phase 8: Model 1 (XGB/GBM) 5-fold OOF predictions
[build]     

## 6. Train & evaluate Model 2 (report: PR-AUC, recall, per-anomaly-type recall)

In [8]:
import os
os.chdir(os.path.join(PROJECT, 'code'))
!python train_model2.py output

[clean] 60120 -> 59910 rows after dedup / bad-amount drop

=== Model 2: XGBoost (scale_pos_weight) ===
PR-AUC (avg precision): 0.815
Precision: 0.482  Recall: 0.869  F1: 0.620

Confusion matrix [ [TN FP] [FN TP] ]:
[[13350   786]
 [  110   732]]

Classification report:
              precision    recall  f1-score   support

           0      0.992     0.944     0.968     14136
           1      0.482     0.869     0.620       842

    accuracy                          0.940     14978
   macro avg      0.737     0.907     0.794     14978
weighted avg      0.963     0.940     0.948     14978

Per-anomaly-type recall (caught / total in test set):
  detention_padding   :  278/ 278 = 1.000
  toll_inflation      :  260/ 275 = 0.945
  weight_discrepancy  :  194/ 289 = 0.671

Top features:
  cost_mismatch                    0.457
  commercial_delay_days            0.227
  model_a_predicted_cost           0.072
  actual_billed_amount             0.063
  true_delay_days                  0.059
  v

## 7. Export deployable model files
Trains final Model 1 + Model 2 on the rebuilt data and writes artifacts to
`code/output/models/` (`.joblib` + xgboost native `.json` + feature schema +
metadata + a runnable `predict_example.py`).

In [9]:
import os
os.chdir(os.path.join(PROJECT, 'code'))
!python export_models.py --outdir output --seed 42
print(sorted(os.listdir('output/models')))

=== Model 1: freight cost regressor ===
  holdout MAE=Rs 456.9  MAPE=1.67%
=== Model 2: invoice risk classifier ===
[clean] invoices 60120 -> 59910 rows
  PR-AUC=0.815 recall=0.869 precision=0.482
  per-type recall: {'detention_padding': 1.0, 'toll_inflation': 0.945, 'weight_discrepancy': 0.671}

Exported artifacts to output/models/
   metadata.json
   model_1_features.json
   model_1_freight_cost.joblib
   model_1_freight_cost.json
   model_2_features.json
   model_2_invoice_risk.joblib
   model_2_invoice_risk.json
   predict_example.py
['metadata.json', 'model_1_features.json', 'model_1_freight_cost.joblib', 'model_1_freight_cost.json', 'model_2_features.json', 'model_2_invoice_risk.joblib', 'model_2_invoice_risk.json', 'predict_example.py']


## 8. (Optional) SARIMA forecaster on the fuel & weather series

In [10]:
import os
os.chdir(os.path.join(PROJECT, 'code'))
!python forecast_series.py --fuel ../real_cache/fuel_prices.csv \
    --weather ../real_cache/weather_cache.csv --outdir ../real_cache/forecasts

=== FUEL (per-state daily diesel, ARIMA(1,1,1)) ===
  [fuel] Andhra Pradesh   MAE=0.000  sMAPE=0.00%  last=95.15  test_sd=0.000
  [fuel] Delhi            MAE=0.000  sMAPE=0.00%  last=87.67  test_sd=0.000
  [fuel] Gujarat          MAE=0.000  sMAPE=0.00%  last=86.95  test_sd=0.000
  [fuel] Haryana          MAE=0.000  sMAPE=0.00%  last=89.45  test_sd=0.000
  [fuel] Karnataka        MAE=0.000  sMAPE=0.00%  last=91.65  test_sd=0.000
  [fuel] Madhya Pradesh   MAE=0.000  sMAPE=0.00%  last=91.95  test_sd=0.000
  [fuel] Maharashtra      MAE=0.000  sMAPE=0.00%  last=90.03  test_sd=0.000
  [fuel] Punjab           MAE=0.000  sMAPE=0.00%  last=90.05  test_sd=0.000
  [fuel] Rajasthan        MAE=0.000  sMAPE=0.00%  last=92.85  test_sd=0.000
  [fuel] Tamil Nadu       MAE=0.000  sMAPE=0.00%  last=92.39  test_sd=0.000
  [fuel] Telangana        MAE=0.000  sMAPE=0.00%  last=93.95  test_sd=0.000
  [fuel] Uttar Pradesh    MAE=0.000  sMAPE=0.00%  last=88.65  test_sd=0.000
  [fuel] West Bengal      MAE=0.000 

d:\chirag\bundle\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
d:\chirag\bundle\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


## 9. Archive the exported model files

In [11]:
import shutil, os
SRC = os.path.join(PROJECT, 'code', 'output', 'models')
ZIP_OUT = os.path.join(PROJECT, 'chi_freight_models')
shutil.make_archive(ZIP_OUT, 'zip', SRC)
print('Created models zip archive at:', ZIP_OUT + '.zip')

Created models zip archive at: d:\chirag\bundle\chi_freight_models.zip


## 10. Sanity-check inference (the exact call a website would make)

In [12]:
import os
os.chdir(os.path.join(PROJECT, 'code', 'output', 'models'))
!python predict_example.py

predicted should-cost: Rs 25,499
invoice risk proba=0.947  flag_for_review=1


## 11. Website integration notes
- **Load once** at server start: `M1 = joblib.load('model_1_freight_cost.joblib')`,
  `M2 = joblib.load('model_2_invoice_risk.joblib')` (+ their `*_features.json`).
- **Model 1 input** (build the row exactly as `predict_example.py` does): numeric
  `distance_km, ideal_days, quoted_days, billable_weight_kg, expected_fuel_price,
  expected_weather_score` + one-hot `product_category` + one-hot `truck_type`
  (truck derived from `weight_kg`). Follow `model_1_features.json['column_order']`.
- **Model 2 input**: the 9 features in `model_2_features.json['features']`; flag when
  `proba >= threshold` (0.5). `model_a_predicted_cost` should be Model 1's output.
- **Portable option**: the xgboost native `.json` boosters load in any language with
  an xgboost binding (Python/Java/Node), independent of the sklearn version.
- Wrap `predict_example.py`'s two functions in a Flask/FastAPI endpoint.